In [1]:
import pytesseract
from pdf2image import convert_from_path
from jiwer import cer, wer
import pandas as pd
from pathlib import Path

TESSERACT_CONFIG = r'--oem 3 --psm 6'

def normalize(text):
    return " ".join(text.lower().split())

def ocr_pdf(pdf_path, dpi=200):
    pages = convert_from_path(pdf_path, dpi=dpi)
    extracted_text = ""
    for page in pages:
        text = pytesseract.image_to_string(page, config=TESSERACT_CONFIG)
        extracted_text += text + "\n"
    return extracted_text

In [2]:
dataset_path = Path("dataset")
results = []

for pdf_path in dataset_path.glob("*.pdf"):
    gt_path = pdf_path.with_suffix(".txt")
    if not gt_path.exists():
        continue

    try:
        ground_truth = normalize(gt_path.read_text(encoding="utf-8"))
        ocr_text = normalize(ocr_pdf(pdf_path))
    except Exception as e:
        print(f"Error on {pdf_path.name}: {e}")
        continue

    results.append({
        "file": pdf_path.name,
        "CER": cer(ground_truth, ocr_text),
        "WER": wer(ground_truth, ocr_text)
    })

df = pd.DataFrame(results)

In [3]:
print("Average CER:", df["CER"].mean())
print("Average WER:", df["WER"].mean())
print(df.sort_values("WER", ascending=False))

Average CER: 0.10793473271187017
Average WER: 0.14045997180134764
                          file       CER       WER
67   motion_civil_clean_06.pdf  0.510774  0.544843
80   motion_civil_clean_01.pdf  0.502262  0.526559
66   motion_civil_clean_04.pdf  0.499659  0.524444
46   motion_civil_clean_09.pdf  0.502116  0.519630
70   motion_civil_clean_07.pdf  0.480743  0.511983
..                         ...       ...       ...
10    civil_order_clean_08.pdf  0.003193  0.030568
107   civil_order_clean_04.pdf  0.003155  0.030568
95    civil_order_clean_07.pdf  0.003183  0.030303
106   civil_order_clean_10.pdf  0.003183  0.030172
84    civil_order_clean_15.pdf  0.002527  0.021739

[108 rows x 3 columns]
